In [ ]:
# 解决OpenMP库冲突警告
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch as th
import torch.nn as nn
import sys
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import csv

# 获取项目根目录
current_dir = os.path.dirname(os.path.abspath('')) if '__file__' not in globals() else os.path.dirname(os.path.abspath(__file__))
if current_dir == '':
    current_dir = os.getcwd()
project_root = os.path.dirname(current_dir) if 'scripts' in current_dir else current_dir
sys.path.insert(0, project_root)

from improved_diffusion.unet import UNetModel

print(f"Current directory: {current_dir}")
print(f"Project root: {project_root}")


## 2. 定义配置和辅助类


In [ ]:
class CondUNet(nn.Module):
    """Wrapper for UNet with conditional input"""
    def __init__(self, base_unet):
        super().__init__()
        self.unet = base_unet

    def forward(self, x, t, cond=None, **kwargs):
        if cond is None:
            raise ValueError("cond is required")
        x_in = th.cat([x, cond], dim=1)
        return self.unet(x_in, t, **kwargs)


# 超参数配置：小、中、大
CONFIGS = {
    "small": {
        "num_channels": 8,
        "channel_mult": (1, 1, 1, 2),
        "batch_size": 1,
        "diffusion_steps": 500,
        "learn_sigma": False,
        "use_checkpoint": True,
        "attention": False,
        "num_res_blocks": 1,
    },
    "medium": {
        "num_channels": 16,
        "channel_mult": (1, 1, 2, 2),
        "batch_size": 1,
        "diffusion_steps": 1000,
        "learn_sigma": False,
        "use_checkpoint": True,
        "attention": False,
        "num_res_blocks": 1,
    },
    "large": {
        "num_channels": 32,
        "channel_mult": (1, 2, 2, 2),
        "batch_size": 2,
        "diffusion_steps": 2000,
        "learn_sigma": True,
        "use_checkpoint": True,
        "attention": False,
        "num_res_blocks": 1,
    },
}

# 尺寸列表
VOLUME_SIZES = [64, 96, 128, 256, 512]

print("Configurations loaded:")
for name, config in CONFIGS.items():
    print(f"  {name}: channels={config['num_channels']}, batch={config['batch_size']}")


## 3. 辅助函数


In [ ]:
def count_parameters(model):
    """计算模型参数数量"""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


In [ ]:
def estimate_model_memory(model, batch_size, volume_size, dtype=th.float32):
    """估算模型显存占用（MB）"""
    # 模型参数显存
    param_memory = sum(p.numel() * p.element_size() for p in model.parameters())
    
    # 优化器状态（AdamW需要2倍参数显存：momentum + variance）
    optimizer_memory = param_memory * 2
    
    # 输入数据显存 (B, 3, D, H, W) - 3 channels: x_t(1) + init(1) + mask(1)
    input_memory = batch_size * 3 * volume_size ** 3 * 4  # float32 = 4 bytes
    
    # 输出显存 (B, out_channels, D, H, W)
    out_channels = 2 if model.unet.out_channels == 2 else 1
    output_memory = batch_size * out_channels * volume_size ** 3 * 4
    
    # 中间激活值显存（基于实际测量校准的估算）
    activation_memory = 0
    ch = model.unet.model_channels
    ds = volume_size
    num_res_blocks = model.unet.num_res_blocks
    channel_mult = model.unet.channel_mult
    
    # Input blocks
    for level, mult in enumerate(channel_mult):
        ch_mult = ch * mult
        for _ in range(num_res_blocks):
            activation_memory += batch_size * ch_mult * ds ** 3 * 4
            activation_memory += batch_size * ch_mult * ds ** 3 * 4 * 2.5  # 内部激活值
        if level < len(channel_mult) - 1:
            activation_memory += batch_size * ch_mult * ds ** 3 * 4
            ds //= 2
    
    # Middle block
    ch_middle = ch * channel_mult[-1]
    ds_middle = ds
    activation_memory += batch_size * ch_middle * ds_middle ** 3 * 4 * 3.5
    activation_memory += batch_size * ch_middle * ds_middle ** 3 * 4 * 3.5
    
    # Output blocks
    for level, mult in enumerate(reversed(channel_mult)):
        ch_mult = ch * mult
        if level > 0:
            ch_concat = ch_mult * 2
        else:
            ch_concat = ch_mult
        for _ in range(num_res_blocks + 1):
            activation_memory += batch_size * ch_concat * ds ** 3 * 4
            activation_memory += batch_size * ch_concat * ds ** 3 * 4 * 2.5
        if level < len(channel_mult) - 1:
            activation_memory += batch_size * ch_concat * ds ** 3 * 4
            ds *= 2
    
    # Checkpoint因子
    if model.unet.use_checkpoint:
        checkpoint_factor = 0.6
    else:
        checkpoint_factor = 1.0
    
    # 梯度显存
    gradient_memory = param_memory
    
    # 反向传播激活值
    backward_activation_memory = activation_memory * 0.6 * checkpoint_factor
    
    # 开销因子
    overhead_factor = 1.35
    
    # 经验校准因子
    # 根据用户反馈，96尺寸实际需要约20GB，但估算仍然被低估
    # 进一步增加校准因子，使其更接近实际值
    # 基础校准因子从25.0增加到30.0，以更准确地匹配实际显存需求
    base_calibration = 30.0  # 从25.0增加到30.0
    size_multiplier = (volume_size / 64) ** 1.5
    empirical_calibration_factor = base_calibration * size_multiplier
    
    # 对于更大的尺寸，额外增加校准因子（因为内存碎片化更严重）
    if volume_size >= 256:
        empirical_calibration_factor *= 1.2  # 256及以上尺寸额外增加20%
    elif volume_size >= 128:
        empirical_calibration_factor *= 1.1  # 128尺寸额外增加10%
    
    # 总显存（MB）
    base_memory = (
        param_memory + 
        optimizer_memory + 
        input_memory + 
        output_memory + 
        activation_memory * checkpoint_factor + 
        backward_activation_memory + 
        gradient_memory
    )
    
    total_memory = base_memory * overhead_factor * empirical_calibration_factor / (1024 ** 2)
    
    activation_memory_mb = activation_memory * checkpoint_factor / (1024 ** 2)
    backward_activation_memory_mb = backward_activation_memory / (1024 ** 2)
    
    return {
        "param_memory_mb": param_memory / (1024 ** 2),
        "optimizer_memory_mb": optimizer_memory / (1024 ** 2),
        "data_memory_mb": (input_memory + output_memory) / (1024 ** 2),
        "activation_memory_mb": activation_memory_mb,
        "backward_activation_memory_mb": backward_activation_memory_mb,
        "gradient_memory_mb": gradient_memory / (1024 ** 2),
        "overhead_mb": (total_memory - total_memory / overhead_factor) * 1024,
        "total_memory_mb": total_memory,
        "total_memory_gb": total_memory / 1024,
    }


## 4. 可视化函数


In [ ]:
## 4.1 更新可视化函数（添加数值标签）


In [ ]:
# 更新plot_3d_bars函数：在每个柱子上方添加数值标签
# 注意：这会覆盖之前定义的plot_3d_bars函数
def plot_3d_bars(ax, results, sizes, configs, value_key, zlabel, title, is_memory=False):
    """绘制3D柱状图（更新版本：显示所有柱子的数值标签）"""
    # 创建数据矩阵
    data_matrix = np.zeros((len(configs), len(sizes)))
    
    # 更新config_map以匹配新的顺序：large-medium-small
    config_map = {"large": 0, "medium": 1, "small": 2}
    size_map = {size: i for i, size in enumerate(sizes)}
    
    for r in results:
        if r["config"] in config_map and r["size"] in size_map:
            config_idx = config_map[r["config"]]
            size_idx = size_map[r["size"]]
            data_matrix[config_idx, size_idx] = r[value_key]
    
    # 设置坐标
    x_pos = np.arange(len(sizes))
    y_pos = np.arange(len(configs))
    x_pos, y_pos = np.meshgrid(x_pos, y_pos)
    
    # 柱状图的位置和大小
    dx = dy = 0.6
    dz = data_matrix.flatten()
    
    # 创建颜色映射
    if dz.max() > 0:
        colors = plt.cm.viridis(dz / dz.max())
    else:
        colors = plt.cm.viridis(np.zeros_like(dz))
    
    # 绘制柱状图
    ax.bar3d(
        x_pos.flatten(), 
        y_pos.flatten(), 
        np.zeros_like(dz),
        dx, dy, dz,
        color=colors,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    
    # 设置标签 - 大幅增加labelpad以确保标签可见
    ax.set_xlabel('Volume Size', fontsize=12, labelpad=20)
    ax.set_ylabel('Config', fontsize=12, labelpad=20)
    # 对于显存图，特别增加z轴labelpad，并调整视角使标签可见
    if is_memory:
        ax.set_zlabel(zlabel, fontsize=12, labelpad=30)
        # 调整视角，使z轴标签更容易看到
        ax.view_init(elev=25, azim=50)
    else:
        ax.set_zlabel(zlabel, fontsize=12, labelpad=20)
        ax.view_init(elev=20, azim=45)
    ax.set_title(title, fontsize=14, pad=25)
    
    # 设置刻度
    ax.set_xticks(range(len(sizes)))
    ax.set_xticklabels([f"{s}³" for s in sizes])
    ax.set_yticks(range(len(configs)))
    ax.set_yticklabels([c.upper() for c in configs])
    
    # 调整轴的边距
    ax.dist = 12  # 增加相机距离，使标签有更多空间
    
    # 在每个柱子上方添加数值标签
    for i in range(len(sizes)):
        for j in range(len(configs)):
            value = data_matrix[j, i]
            if value > 0:
                # 根据数值大小格式化标签
                if value_key == "total_memory_gb":
                    if value >= 1000:
                        label = f'{value/1000:.1f}TB'
                    elif value >= 1:
                        label = f'{value:.1f}GB'
                    else:
                        label = f'{value*1024:.0f}MB'
                elif value_key == "total_params_m":
                    if value >= 1000:
                        label = f'{value/1000:.2f}B'
                    elif value >= 1:
                        label = f'{value:.2f}M'
                    else:
                        label = f'{value*1000:.0f}K'
                else:
                    label = f'{value:.2f}'
                
                # 在柱子顶部上方添加标签
                z_pos = value + value * 0.05  # 在柱子顶部上方5%的位置
                ax.text(i, j, z_pos, label, 
                       fontsize=9, ha='center', va='bottom',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7, edgecolor='gray', linewidth=0.5))

print("✓ plot_3d_bars函数已更新：现在会在每个柱子上方显示数值标签")


In [ ]:
def plot_3d_bars(ax, results, sizes, configs, value_key, zlabel, title, is_memory=False):
    """绘制3D柱状图"""
    # 创建数据矩阵
    data_matrix = np.zeros((len(configs), len(sizes)))
    
    # 更新config_map以匹配新的顺序：large-medium-small
    config_map = {"large": 0, "medium": 1, "small": 2}
    size_map = {size: i for i, size in enumerate(sizes)}
    
    for r in results:
        if r["config"] in config_map and r["size"] in size_map:
            config_idx = config_map[r["config"]]
            size_idx = size_map[r["size"]]
            data_matrix[config_idx, size_idx] = r[value_key]
    
    # 设置坐标
    x_pos = np.arange(len(sizes))
    y_pos = np.arange(len(configs))
    x_pos, y_pos = np.meshgrid(x_pos, y_pos)
    
    # 柱状图的位置和大小
    dx = dy = 0.6
    dz = data_matrix.flatten()
    
    # 创建颜色映射
    if dz.max() > 0:
        colors = plt.cm.viridis(dz / dz.max())
    else:
        colors = plt.cm.viridis(np.zeros_like(dz))
    
    # 绘制柱状图
    ax.bar3d(
        x_pos.flatten(), 
        y_pos.flatten(), 
        np.zeros_like(dz),
        dx, dy, dz,
        color=colors,
        alpha=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    
    # 设置标签 - 大幅增加labelpad以确保标签可见
    ax.set_xlabel('Volume Size', fontsize=12, labelpad=20)
    ax.set_ylabel('Config', fontsize=12, labelpad=20)
    # 对于显存图，特别增加z轴labelpad，并调整视角使标签可见
    if is_memory:
        ax.set_zlabel(zlabel, fontsize=12, labelpad=30)
        # 调整视角，使z轴标签更容易看到
        ax.view_init(elev=25, azim=50)
    else:
        ax.set_zlabel(zlabel, fontsize=12, labelpad=20)
        ax.view_init(elev=20, azim=45)
    ax.set_title(title, fontsize=14, pad=25)
    
    # 设置刻度
    ax.set_xticks(range(len(sizes)))
    ax.set_xticklabels([f"{s}³" for s in sizes])
    ax.set_yticks(range(len(configs)))
    ax.set_yticklabels([c.upper() for c in configs])
    
    # 调整轴的边距
    ax.dist = 12  # 增加相机距离，使标签有更多空间
    
    # 添加数值标签
    if value_key == "total_memory_gb" and dz.max() < 200:
        for i in range(len(sizes)):
            for j in range(len(configs)):
                value = data_matrix[j, i]
                if value > 0:
                    ax.text(i, j, value, f'{value:.1f}', 
                           fontsize=8, ha='center', va='bottom')


In [ ]:
def create_3d_visualization(results):
    """创建3D柱状图可视化"""
    # 准备数据
    sizes = sorted(set(r["size"] for r in results))
    configs = ["large", "medium", "small"]
    
    print(f"\nGenerating 3D visualization...")
    print(f"  Sizes: {sizes}")
    print(f"  Configs: {configs}")
    print(f"  Results count: {len(results)}")
    
    # 创建两个图表，使用更大的figsize和更多的边距
    fig = plt.figure(figsize=(24, 10))
    
    # 使用GridSpec来更好地控制子图位置和间距
    from matplotlib.gridspec import GridSpec
    gs = GridSpec(1, 2, figure=fig, left=0.08, right=0.95, top=0.92, bottom=0.08, wspace=0.3)
    
    # 图表1：显存需求（GB）
    ax1 = fig.add_subplot(gs[0], projection='3d')
    plot_3d_bars(ax1, results, sizes, configs, "total_memory_gb", 
                 "Memory (GB)", "Memory Budget", is_memory=True)
    
    # 图表2：参数规模（M）
    ax2 = fig.add_subplot(gs[1], projection='3d')
    plot_3d_bars(ax2, results, sizes, configs, "total_params_m",
                 "Parameters (M)", "Parameter Count", is_memory=False)
    
    # 在Jupyter中显示图表
    plt.show()
    
    # 保存图表，使用更大的边距
    plot_path = os.path.join(current_dir, "outpaint_3d_budget_visualization.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight', pad_inches=0.5)
    print(f"\n✓ 3D visualization saved to: {plot_path}")


## 5. 模型创建函数


In [ ]:
def create_model(volume_size, config):
    """创建模型"""
    attention_resolutions = tuple() if not config["attention"] else (volume_size // 16, volume_size // 8)
    out_channels = 2 if config["learn_sigma"] else 1
    
    base_unet = UNetModel(
        in_channels=3,  # x_t + init + mask
        model_channels=config["num_channels"],
        out_channels=out_channels,
        num_res_blocks=config["num_res_blocks"],
        attention_resolutions=attention_resolutions,
        dropout=0.0,
        channel_mult=config["channel_mult"],
        num_classes=None,
        use_checkpoint=config["use_checkpoint"],
        num_heads=1,
        num_heads_upsample=-1,
        use_scale_shift_norm=True,
        dims=3,  # 3D
    )
    
    model = CondUNet(base_unet)
    return model


## 6. 主估算函数


In [ ]:
def estimate_budget():
    """估算所有配置的预算"""
    results = []
    
    print("=" * 100)
    print("3D Outpainting Model Budget Estimator")
    print("=" * 100)
    print()
    
    for size in VOLUME_SIZES:
        print(f"\n{'='*100}")
        print(f"Volume Size: {size}³")
        print(f"{'='*100}")
        
        for config_name, config in CONFIGS.items():
            try:
                # 创建模型
                model = create_model(size, config)
                
                # 计算参数
                total_params, trainable_params = count_parameters(model)
                total_params_m = total_params / 1e6
                
                # 估算显存
                memory_info = estimate_model_memory(
                    model, 
                    config["batch_size"], 
                    size
                )
                
                # 存储结果
                result = {
                    "size": size,
                    "config": config_name,
                    "num_channels": config["num_channels"],
                    "channel_mult": config["channel_mult"],
                    "batch_size": config["batch_size"],
                    "diffusion_steps": config["diffusion_steps"],
                    "learn_sigma": config["learn_sigma"],
                    "use_checkpoint": config["use_checkpoint"],
                    "attention": config["attention"],
                    "total_params": total_params,
                    "total_params_m": total_params_m,
                    **memory_info,
                }
                results.append(result)
                
                # 打印结果
                print(f"\n  Config: {config_name.upper()}")
                print(f"    Parameters: {total_params_m:.2f}M ({total_params:,})")
                print(f"    Memory Breakdown:")
                print(f"      - Model Parameters: {memory_info['param_memory_mb']:.2f} MB")
                print(f"      - Optimizer States: {memory_info['optimizer_memory_mb']:.2f} MB")
                print(f"      - Data (Input+Output): {memory_info['data_memory_mb']:.2f} MB")
                checkpoint_factor_used = 0.6 if config["use_checkpoint"] else 1.0
                base_calibration = 30.0
                calibration_factor = base_calibration * ((size / 64) ** 1.5)
                # 应用额外的尺寸校准
                if size >= 256:
                    calibration_factor *= 1.2
                elif size >= 128:
                    calibration_factor *= 1.1
                print(f"      - Activations (forward, checkpoint: {checkpoint_factor_used:.1f}): {memory_info['activation_memory_mb']:.2f} MB")
                print(f"      - Activations (backward): {memory_info.get('backward_activation_memory_mb', 0):.2f} MB")
                print(f"      - Gradients: {memory_info['gradient_memory_mb']:.2f} MB")
                print(f"      - Overhead (PyTorch/CUDA): {memory_info.get('overhead_mb', 0):.2f} MB")
                print(f"      - Empirical Calibration Factor: {calibration_factor:.1f}x")
                print(f"    Total Memory: {memory_info['total_memory_gb']:.2f} GB ({memory_info['total_memory_mb']:.2f} MB)")
                
            except Exception as e:
                print(f"\n  Config: {config_name.upper()} - ERROR: {e}")
                import traceback
                traceback.print_exc()
                continue
    
    # 生成汇总表格
    print("\n" + "=" * 100)
    print("SUMMARY TABLE")
    print("=" * 100)
    print(f"{'Size':<6} {'Config':<8} {'Params(M)':<12} {'Memory(GB)':<12} {'Batch':<6} {'Channels':<10} {'ChMult':<20}")
    print("-" * 100)
    
    for r in results:
        ch_mult_str = str(r["channel_mult"]).replace(" ", "")
        print(f"{r['size']:<6} {r['config']:<8} {r['total_params_m']:<12.2f} {r['total_memory_gb']:<12.2f} "
              f"{r['batch_size']:<6} {r['num_channels']:<10} {ch_mult_str:<20}")
    
    # 保存到CSV
    csv_path = os.path.join(current_dir, "outpaint_3d_budget_results.csv")
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=[
            'size', 'config', 'num_channels', 'channel_mult', 'batch_size',
            'diffusion_steps', 'learn_sigma', 'use_checkpoint', 'attention',
            'total_params', 'total_params_m', 'param_memory_mb', 'optimizer_memory_mb',
            'data_memory_mb', 'activation_memory_mb', 'backward_activation_memory_mb',
            'gradient_memory_mb', 'overhead_mb', 'total_memory_mb', 'total_memory_gb'
        ])
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\n\nResults saved to: {csv_path}")
    
    # 生成3D可视化图表
    create_3d_visualization(results)
    
    return results


In [ ]:
# 设置设备（不需要实际GPU，只需要计算）
th.set_default_dtype(th.float32)

# 执行估算
results = estimate_budget()


## 8. 查看结果（可选调试）


In [ ]:
# 查看特定配置的结果
# 例如：查看96尺寸large配置的结果
if 'results' in locals():
    for r in results:
        if r['size'] == 96 and r['config'] == 'large':
            print(f"96³ Large Config:")
            print(f"  Parameters: {r['total_params_m']:.2f}M")
            print(f"  Memory: {r['total_memory_gb']:.2f} GB")
            print(f"  Breakdown:")
            print(f"    - Params: {r['param_memory_mb']:.2f} MB")
            print(f"    - Optimizer: {r['optimizer_memory_mb']:.2f} MB")
            print(f"    - Data: {r['data_memory_mb']:.2f} MB")
            print(f"    - Activations (forward): {r['activation_memory_mb']:.2f} MB")
            print(f"    - Activations (backward): {r['backward_activation_memory_mb']:.2f} MB")
            print(f"    - Gradients: {r['gradient_memory_mb']:.2f} MB")
            break


In [ ]:
# 创建DataFrame以便更好地查看和分析结果
try:
    import pandas as pd
    
    if 'results' in locals():
        df = pd.DataFrame(results)
        print("Results DataFrame:")
        print(df[['size', 'config', 'total_params_m', 'total_memory_gb', 'batch_size', 'num_channels']])
        
        # 按显存需求排序
        print("\nSorted by Memory (GB):")
        print(df.sort_values('total_memory_gb', ascending=False)[['size', 'config', 'total_memory_gb', 'total_params_m']])
except ImportError:
    print("pandas not available, skipping DataFrame view")


## 9. 注意事项

**注意：** 显存估算为近似值，实际值可能因以下因素而有所不同：
- 训练过程中的实际激活模式
- PyTorch内存分配器的行为
- CUDA内存碎片
- 数据加载、日志记录等的额外开销
